# 03 — Materialize only the selected NeMo data

NeMo training requires local audio paths. This stage downloads **only** the rows selected in the manifest and writes WAV files plus JSONL manifests for Train, Validation, and Test. The rest of EveryAyah remains undownloaded.

In [ ]:
from pathlib import Path
import os

# Each notebook may open in a fresh Colab runtime, so mount Drive before any
# path check rather than relying on a previous notebook's session.
from google.colab import drive
DRIVE_ROOT = Path("/content/drive/MyDrive")
if not DRIVE_ROOT.exists():
    drive.mount("/content/drive")

# Place the *contents* of this repository in this Google Drive folder, or edit
# this one variable to match the folder you chose.
PROJECT_DIR = DRIVE_ROOT / "quran-fastconformer-colab"
assert PROJECT_DIR.exists(), f"Project directory not found: {PROJECT_DIR}"
os.chdir(PROJECT_DIR)
print("Working directory:", Path.cwd())


In [ ]:
!python run_colab_setup.py --config configs/fastconformer_quran.yaml --materialize


In [ ]:
import json
from pathlib import Path
from src.data import manifest_summary

manifest = json.loads(Path("artifacts/manifests/experiment_manifest.json").read_text(encoding="utf-8"))
summary = manifest_summary(manifest)
summary


In [ ]:
# Scientific integrity guard: every reciter must be owned by one split only.
reciter_sets = {split: set(info["reciters"]) for split, info in summary.items() if split != "integrity"}
for left, right in (("train", "validation"), ("train", "test"), ("validation", "test")):
    assert not (reciter_sets[left] & reciter_sets[right]), f"Reciter leakage: {left}/{right}"
assert summary["integrity"]["reciter_leakage_count"] == 0
print("Passed: all validation/test reciters are unseen during training.")
